# Experimental submission — same models, parallel DICOM preparation

Attach only the final package from the successful **03c parallel timing run** and the **RSNA competition**. The package must contain `rsna3-inference/`, its fifteen complete weights, the offline wheels, the new helper and the measured parallel admission profile. A private dataset with that folder is accepted.

Select **T4 ×2**, Internet **off**. Use **Save Version → Save & Run All** to validate the visible examples, then **Submit to Competition** and choose `submission.csv`. Hidden inference is a separate execution on the hidden test input; three visible rows only check execution.

The notebook checks the measured helper/worker settings, then reads all actual test studies and applies the measured eight-hour admission check. Pixel rendering and the trained ensemble are unchanged. This remains an experimental submission because its gold AUC was below the existing frozen baseline; keep that baseline available as your main candidate.


In [ ]:
import time
NOTEBOOK_STARTED = time.monotonic()
from pathlib import Path
Path('/kaggle/working/submission.csv').unlink(missing_ok=True)
import importlib.util
import json
import subprocess
import sys
BUNDLE_ROOT = None
if BUNDLE_ROOT is None:
    choices = set()
    # Known Kaggle input layouts; never recurse through the DICOM directory.
    for pattern in ('*/manifest.json', '*/rsna3-inference/manifest.json',
                    'datasets/*/*/manifest.json', 'datasets/*/*/rsna3-inference/manifest.json',
                    'notebooks/*/*/manifest.json', 'notebooks/*/*/rsna3-inference/manifest.json'):
        for p in Path('/kaggle/input').glob(pattern):
            try:
                if json.loads(p.read_text()).get('format') == 'rsna3-full-models-v1':
                    choices.add(p.parent)
            except (OSError, ValueError):
                pass
    choices = sorted(choices)
    if len(choices) != 1:
        raise ValueError(f'Choisir BUNDLE_ROOT parmi {choices}')
    BUNDLE_ROOT = choices[0]
BUNDLE_ROOT = Path(BUNDLE_ROOT)
# Optional Python ABI patch; only its shallow dataset metadata is searched.
WHEELS_PATCH_ROOT = None
patch_manifests = list(Path('/kaggle/input').glob('*/wheels_manifest.json'))
patch_manifests += list(Path('/kaggle/input').glob('datasets/*/*/wheels_manifest.json'))
patch_roots = [p.parent for p in patch_manifests
               if json.loads(p.read_text()).get('format') == 'rsna3-offline-wheels-patch-v1']
if len(patch_roots) > 1:
    raise ValueError(f'Attach a single wheels patch: {patch_roots}')
if patch_roots:
    WHEELS_PATCH_ROOT = patch_roots[0]
BOOTSTRAP_ROOT = WHEELS_PATCH_ROOT or BUNDLE_ROOT
module_spec = importlib.util.spec_from_file_location('kaggle_bootstrap', BOOTSTRAP_ROOT / 'kaggle_bootstrap.py')
bootstrap = importlib.util.module_from_spec(module_spec)
module_spec.loader.exec_module(bootstrap)
if WHEELS_PATCH_ROOT is None:
    bootstrap.install(BUNDLE_ROOT)
else:
    bootstrap.install(BUNDLE_ROOT, wheels_root=WHEELS_PATCH_ROOT)
WORK = Path('/kaggle/working')

from rsna3.common import read_json, require
from rsna3.inference import load_bundle
manifest = load_bundle(BUNDLE_ROOT)
require(manifest.get('required_execution_pipeline') == 'parallel-study-cache-v1',
        'Attach the measured output of 03c, including its parallel helper.')
require((BUNDLE_ROOT / 'rsna3_parallel_inference.py').is_file(), 'Missing packaged parallel helper')
from rsna3_parallel_inference import submit_parallel, execution_contract, SCOPE
execution = manifest['admission_profile']['execution']
CPU_WORKERS, CACHE_BATCH_SIZE = execution['workers'], execution['batch_size']
require(manifest['admission_profile']['scope'] == SCOPE and
        execution_contract(CPU_WORKERS, CACHE_BATCH_SIZE) == execution,
        'The packaged helper differs from the measured execution path.')


In [ ]:
COMPETITION_ROOT = None
if COMPETITION_ROOT is None:
    choices = set()
    for pattern in ('*/test_series.csv', 'competitions/*/test_series.csv', 'datasets/*/*/test_series.csv'):
        for p in Path('/kaggle/input').glob(pattern):
            if (p.parent / 'test.csv').is_file():
                choices.add(p.parent)
    require(len(choices) == 1, f'Attach one RSNA competition input; found: {sorted(choices)}')
    COMPETITION_ROOT = next(iter(choices))
COMPETITION_ROOT = Path(COMPETITION_ROOT)

def dicom_directory(split):
    choices = [COMPETITION_ROOT / name for name in (f'{split}_series', f'{split}_images', split)
               if (COMPETITION_ROOT / name).is_dir()]
    require(len(choices) == 1, f'Choose the DICOM folder for {split}: {choices}')
    return choices[0]


In [ ]:
# Use every actual test UID and preserve the same fifteen-model ensemble.
remaining_hours = min(8., 9. - (time.monotonic() - NOTEBOOK_STARTED) / 3600 - .5)
require(remaining_hours > 0, 'Insufficient runtime remains after setup')
output_csv = WORK / 'submission.csv'
receipt = submit_parallel(BUNDLE_ROOT, COMPETITION_ROOT / 'test.csv',
    COMPETITION_ROOT / 'test_series.csv', dicom_directory('test'),
    WORK / 'rsna3-test', output_csv, max_hours=remaining_hours,
    workers=CPU_WORKERS, batch_size=CACHE_BATCH_SIZE)
print(json.dumps(receipt, indent=2))


In [ ]:
import pandas as pd
submission = pd.read_csv(output_csv)
print(submission.shape)
display(submission.head())
print((WORK / 'rsna3-test/submission_receipt.json').read_text())
